# Can we predict whether a student passes the closing quiz?

**Question:** from what we know about a session (`prior_score`, `session_minutes`, `hints_used`) and the student's context (`enrol_year`, `school_code`), can we predict `passed_quiz`?

**Design decisions up front**
- Rows are *sessions*, not students: 12,453 sessions from 1,800 students (about 7 each). Sessions from one student are correlated, so a random row split puts the same student in train and test and can flatter the model. **All evaluation uses student-grouped splits**, which mimics predicting for a student the model hasn't seen.
- `student_key` is an identifier, not a feature.
- Metrics: ROC AUC (ranking), plus log loss / Brier (probability quality). The base pass rate is ~42%, so accuracy alone is a weak yardstick.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_predict, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, roc_curve, log_loss, brier_score_loss, accuracy_score
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
pd.set_option('display.width', 140)
df = pd.read_csv('data.csv'); cols = pd.read_csv('columns.csv')
cols

## 1. Data checks

In [ ]:
print(df.shape, '| students:', df.student_key.nunique(), '| duplicate rows:', df.duplicated().sum())
print('missing values:', int(df.isna().sum().sum()))
print(df.describe().T[['mean','std','min','50%','max']].round(2))
print('\nsessions per student:'); print(df.groupby('student_key').size().describe().round(1))
print('\npass rate:', df.passed_quiz.mean().round(3))
print('prior_score > 100:', (df.prior_score > 100).sum())

In [ ]:
g = df.groupby('student_key')
print('prior_score SD within student (mean):', g.prior_score.std().mean().round(2), '| overall SD:', df.prior_score.std().round(2))
print('enrol_year / school_code distinct values per student (max):', g.enrol_year.nunique().max(), g.school_code.nunique().max())
dev = df.prior_score - g.prior_score.transform('mean')
print('corr(passed, within-student prior_score deviation) =', round(np.corrcoef(dev, df.passed_quiz)[0,1], 4))
s = g.passed_quiz.agg(['mean', 'size'])
p0 = df.passed_quiz.mean()
print('variance of student pass rates       :', round(s['mean'].var(), 4))
print('expected if all students were alike  :', round((p0*(1-p0)/s['size']).mean(), 4))

Things worth noting:
- `prior_score` is described as a *placement test* score, yet it **varies between sessions of the same student** (within-student SD ~0.3 vs ~11.6 overall). It behaves like a near-constant student attribute plus small jitter, and the jitter has no relationship with the outcome.
- 19 `prior_score` values exceed 100 and a few sessions are very long (max 204 min). I left them as they are (few, and the models used are not sensitive to them), but the >100 scores are worth confirming with the data owner.
- `enrol_year` and `school_code` are constant within a student.
- Student pass rates vary **far more** than chance would produce, so students differ a lot in how likely they are to pass. The question is whether the available columns capture that.

## 2. Exploratory look at the features

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for a, c in zip(ax, ['prior_score', 'session_minutes', 'hints_used']):
    x = df[c] if c == 'hints_used' else pd.qcut(df[c], 10, duplicates='drop')
    r = df.groupby(x, observed=True).passed_quiz.agg(['mean', 'size'])
    se = np.sqrt(r['mean']*(1-r['mean'])/r['size'])
    xs = np.arange(len(r))
    a.errorbar(xs, r['mean'], 1.96*se, fmt='o-', color='#2a6f97')
    a.set_xticks(xs)
    a.set_xticklabels([str(i) if c == 'hints_used' else f'{i.mid:.0f}' for i in r.index], fontsize=8)
    a.set_title(f'pass rate by {c}' + ('' if c == 'hints_used' else ' (decile mid)'))
    a.set_ylim(0.2, 0.7); a.axhline(p0, ls=':', c='grey')
plt.tight_layout(); plt.show()

In [ ]:
print(df.groupby('enrol_year').passed_quiz.mean().round(3).to_dict())
sch = df.groupby('school_code').passed_quiz.agg(['mean', 'size'])
print('schools:', len(sch), '| pass-rate range', sch['mean'].min().round(2), '-', sch['mean'].max().round(2),
      '| sessions per school', sch['size'].min(), '-', sch['size'].max())

Longer sessions go with higher pass rates, more hints with lower; `prior_score` is close to flat. Effects look modest. (Whether longer sessions *cause* passing is a separate question: a student may stay longer because they are doing well. Here we only care about prediction.)

## 3. Models, evaluated with student-grouped cross-validation

In [ ]:
num = ['prior_score', 'session_minutes', 'hints_used']
allf = num + ['enrol_year', 'school_code']
y = df.passed_quiz.values; groups = df.student_key.values
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
gb = lambda **k: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0, **k)
lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

models = {
 'majority class':                  (DummyClassifier(strategy='prior'), num),
 'logistic (3 session feats)':      (lr(), num),
 'logistic (+year, school)':        (lr(), allf),
 'boosting (3 session feats)':      (gb(), num),
 'boosting (+year, school as cat)': (gb(categorical_features=[4]), allf),
}
rows, oof = [], {}
for name, (m, feats) in models.items():
    p = cross_val_predict(m, df[feats], y, cv=cv, groups=groups, method='predict_proba')[:, 1]
    oof[name] = p
    fa = [roc_auc_score(y[te], p[te]) for _, te in cv.split(df, y, groups)]
    rows.append(dict(model=name, AUC=roc_auc_score(y, p), AUC_fold_sd=np.std(fa),
                     logloss=log_loss(y, p), brier=brier_score_loss(y, p), accuracy=accuracy_score(y, p > 0.5)))
res = pd.DataFrame(rows).set_index('model').round(4)
res

The majority baseline gives AUC 0.5 and the base-rate log loss. Compare the others against that and against the fold-to-fold SD of AUC.

In [ ]:
# Standardised logistic coefficients with student-clustered bootstrap CIs
rng = np.random.default_rng(0)
ids = df.student_key.unique(); by = df.groupby('student_key').indices
Z = StandardScaler().fit_transform(df[num]); coefs = []
for _ in range(200):
    idx = np.concatenate([by[k] for k in rng.choice(ids, len(ids))])
    coefs.append(LogisticRegression(max_iter=1000).fit(Z[idx], y[idx]).coef_[0])
coefs = np.array(coefs)
pd.DataFrame({'coef per SD': coefs.mean(0), 'ci_lo': np.percentile(coefs, 2.5, 0), 'ci_hi': np.percentile(coefs, 97.5, 0)}, index=num).round(3)

In [ ]:
sel = ['logistic (3 session feats)', 'boosting (3 session feats)']
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for n in sel:
    f, t, _ = roc_curve(y, oof[n]); ax[0].plot(f, t, label=f'{n} (AUC {roc_auc_score(y, oof[n]):.3f})')
    fp, mp = calibration_curve(y, oof[n], n_bins=10, strategy='quantile'); ax[1].plot(mp, fp, 'o-', label=n)
ax[0].plot([0, 1], [0, 1], 'k:'); ax[0].set(xlabel='false positive rate', ylabel='true positive rate', title='Out-of-fold ROC'); ax[0].legend(fontsize=8)
ax[1].plot([0, 1], [0, 1], 'k:'); ax[1].set(xlabel='predicted probability', ylabel='observed pass rate', title='Calibration'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 4. Random vs. grouped splits, and the student effect

In [ ]:
# (a) naive random row split: the same students appear in train and test
p_rand = cross_val_predict(gb(), df[num], y, cv=StratifiedKFold(5, shuffle=True, random_state=0), method='predict_proba')[:, 1]

# (b) add the student's pass rate over their *other* sessions (leave-one-out). Not a valid
# deployable feature as built: it uses the student's later sessions too. It is an upper bound.
sm = g.passed_quiz.transform('sum'); n = g.passed_quiz.transform('size')
loo = ((sm - df.passed_quiz) / (n - 1)).where(n > 1, p0)
p_id = cross_val_predict(gb(), df[num].assign(student_loo_rate=loo), y, cv=cv, groups=groups, method='predict_proba')[:, 1]

print(f'3 feats, random row split           AUC {roc_auc_score(y, p_rand):.3f}')
print(f'3 feats, student-grouped split      AUC {roc_auc_score(y, oof["boosting (3 session feats)"]):.3f}')
print(f"+ student's other-session pass rate AUC {roc_auc_score(y, p_id):.3f}  (upper bound, see note)")

The random split is only slightly optimistic here (the features themselves don't identify a student much), but the grouped split is the honest one. More importantly, how a student did in *other* sessions is far more informative than anything in the five columns. The leave-one-out rate uses future sessions as well, so it overstates what is achievable; a real version would use only earlier sessions, and this data has no session date to build that. It does show where the predictive information is.

## 5. Permutation importance (student-grouped hold-out)

In [ ]:
tr, te = next(GroupShuffleSplit(1, test_size=0.25, random_state=1).split(df, y, groups))
m = gb().fit(df.iloc[tr][num], y[tr])
pi = permutation_importance(m, df.iloc[te][num], y[te], scoring='roc_auc', n_repeats=20, random_state=0)
print('hold-out AUC:', round(roc_auc_score(y[te], m.predict_proba(df.iloc[te][num])[:, 1]), 3))
pd.DataFrame({'AUC drop': pi.importances_mean, 'sd': pi.importances_std}, index=num).round(4)

## 6. Conclusions

**Answer: only weakly.** Evaluated on students the model has not seen (5-fold, grouped by student):

| model | AUC | log loss |
|---|---|---|
| majority class | 0.50 | 0.681 |
| logistic, 3 session features | **0.60** | **0.664** |
| logistic, + year, school | 0.60 | 0.665 |
| boosting, 3 session features | 0.59 | 0.672 |
| boosting, + year, school | 0.57 | 0.687 |

(Numbers from the run I did while building this; re-running the notebook should reproduce them, since seeds are fixed.)

- `session_minutes` (longer, more likely to pass) and `hints_used` (more hints, less likely) carry the signal; their bootstrap intervals exclude zero. `prior_score`, `enrol_year` and `school_code` add nothing detectable.
- The simplest model is the best. Boosting overfits and is *worse* than logistic regression, and adding school/year hurts it, so there is no hidden non-linearity to chase with these columns.
- An AUC of 0.60 with accuracy ~60% (vs 58% for always predicting "fail") is a real but small signal. I would not use it for decisions about individual students; it could support rough, group-level risk ranking.
- The big missing piece is *which student*: pass rates differ across students far more than chance allows, and a student's pass rate in other sessions lifts AUC to ~0.85 (an upper bound, see section 4). Student history (using only earlier sessions), topic or quiz difficulty would likely help far more than model tuning. This data has no session date or topic.

**Caveats**
- Associations, not causes: longer sessions and fewer hints may reflect how well the student is doing rather than drive it.
- `prior_score` is labelled a placement score yet changes between a student's sessions, and 19 values exceed 100; worth checking how it is recorded.
- No time ordering, so drift can't be tested (pass rate dips a little in 2022-23 enrolment years).
- Permutation importance on a single grouped hold-out is noisy (hold-out AUC 0.55 for boosting); treat it as secondary to the cross-validated table.